# Distribution grid flexiblity in ASSUME: A minimal example.

## Agents Overview

| Agent                  | Motives                         | Actions                                             | Implementation                                   |
|------------------------|----------------------------------|----------------------------------------------------|--------------------------------------------------|
| DSO                    | Congestion minimization          | Define grid fee                                    | Translate load forecast into fee with single function                 |
| Aggregator (incl. EVs) | Profit maximization              | Define loading power (bidirectional option).<br>Constrained by EVs. | Optimization problem (with slack relaxation, later: RL) |
| External generator     | Profit maximization              | Bid on wholesale market                            | Assume generator                                 |
| External demand        | Inelastic / later maybe elastic  | Bid on wholesale market                            | Assume demand                                    |


## Sequential Process Description

### 1. Inputs are provided to the system
- External Demand  
- External Generation  
- Capacity Requests  
- Residual Load  

---

### 2. Forecasts are generated
- **Market Forecast**
  - Uses External Demand  
  - Uses External Generation  

- **EV Forecast**
  - Uses External Generation  
  - What is this used for? Only DA-Market? Maybe unnecessary.

- **Congestion Forecast**
  - Uses Capacity Requests  
  - Uses Residual Load  

---

### 3. Forecasts are distributed to agents
- Market Forecast → DSO  
- EV Forecast → Aggregator  
- Congestion Forecast →  
  - DSO  
  - Aggregator  

---

### 4. Agents and external actors produce outputs
- DSO → Grid Fee  
- Aggregator → Power Schedule  
- External Generator → Market Bids
- External Demand → Market Bids  

---

### 5. Outputs feed into market bidding
- Grid Fee + Market Forecast -> Optimal Power Schedules
- Power Schedule → Market Bids  
- Market Clearing → Congestion


In [1]:
import numpy as np

In [2]:
# Provide Inputs

# MV assets.
external_demand = [8] * 12
external_generation = [10] * 12

# EVs: (ev_id, capacity, max_charging_power)
evs = [(0, 90, 22), (1, 80, 11)]

# Capacity Requests: (ev_id, start_soc, end_soc, start_step, end_step)
capacity_requests = [(0, 53, 100, 0, 11), (1, 20, 80, 0, 6)]

# Residual Load of households.
residual_load_hh1 = [0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.2, 1.0, 1.0, 0.1, 0.0, -2.0]
residual_load_hh2 = [0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.2, 1.0, 1.0, 0.1, 0.0, -2.0]

In [3]:
# Generate Forecasts.
market_forecast = [300, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]
congestion_forecast = [0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.2, 1.0, 1.0, 0.1, 0.0, -2.0]

In [4]:
# DSO: Use congestion forecast and set grid fees.
grid_fees = [x * 25 for x in congestion_forecast]

evs = {0: {'cap': 90, 'p_max': 22}, 1: {'cap': 80, 'p_max': 11}}
requests = {1: {'s_soc': 53, 'e_soc': 100, 's_t': 0, 'e_t': 11},
            0: {'s_soc': 20, 'e_soc': 80, 's_t': 0, 'e_t': 6}}
cost_prediction = np.array(market_forecast) + np.array(grid_fees)
timesteps = range(len(cost_prediction))


In [ ]:
import os

print(os.environ.get('GUROBI_LICENSE_FILE'))
print(os.environ.get('GUROBI_HOME'))

C:\Users\HiWi 2\licenses\gurobi.lic
C:\gurobi1103\win64


In [ ]:
import pyomo.environ as aml
from pyomo.opt import SolverFactory

def optimal_schedule(evs, timesteps, requests, price_forecast):
    # --- Konfiguration ---
    delta_t = 1.0 
    eff = 0.93  # 90% Effizienz (gilt für Laden UND Entladen)

    # --- Daten-Setup ---

    model = aml.ConcreteModel()
    model.EVs = aml.Set(initialize=evs.keys())
    model.T = aml.Set(initialize=timesteps)

    # Variablen: Aufgeteilt in Charge und Discharge (beide >= 0)
    model.p_ch = aml.Var(model.EVs, model.T, domain=aml.NonNegativeReals)
    model.p_ds = aml.Var(model.EVs, model.T, domain=aml.NonNegativeReals)
    model.soc = aml.Var(model.EVs, model.T, domain=aml.NonNegativeReals, bounds=(0, 100))

    # --- Zielfunktion ---
    # Kosten = (Gekaufte Energie - Verkaufte Energie) * Preis
    def obj_rule(model):
        return sum((model.p_ch[ev, t] - model.p_ds[ev, t]) * delta_t * price_forecast[t] 
                for ev in model.EVs for t in model.T)
    model.obj = aml.Objective(rule=obj_rule, sense=aml.minimize)

    # --- Nebenbedingungen ---

    # 1. Leistungslimits (kombiniert für beide Richtungen)
    def max_p_rule(model, ev, t):
        return model.p_ch[ev, t] + model.p_ds[ev, t] <= evs[ev]['p_max']
    model.max_p_con = aml.Constraint(model.EVs, model.T, rule=max_p_rule)

    # 2. Zeitfenster
    # Für einen Request okay. ToDo: erweitere für mehrere Requests (Availability).
    def window_rule(model, ev, t):
        if t < requests[ev]['s_t'] or t > requests[ev]['e_t']:
            return (model.p_ch[ev, t] + model.p_ds[ev, t]) == 0
        return aml.Constraint.Skip
    model.window_con = aml.Constraint(model.EVs, model.T, rule=window_rule)

    # 3. SoC Dynamik mit Verlusten
    def soc_dynamic_rule(model, ev, t):
        # Energie-Änderung in %: (Laden * Effizienz - Entladen / Effizienz)
        soc_gain = (model.p_ch[ev, t] * eff - model.p_ds[ev, t] / eff) * delta_t / evs[ev]['cap'] * 100
        
        if t == requests[ev]['s_t']:
            return model.soc[ev, t] == requests[ev]['s_soc'] + soc_gain
        elif t > requests[ev]['s_t'] and t <= requests[ev]['e_t']:
            return model.soc[ev, t] == model.soc[ev, t-1] + soc_gain
        else:
            if t == 0: return model.soc[ev, t] == requests[ev]['s_soc']
            return model.soc[ev, t] == model.soc[ev, t-1]
    model.soc_con = aml.Constraint(model.EVs, model.T, rule=soc_dynamic_rule)

    # 4. Ziel-SoC
    def target_soc_rule(model, ev):
        return model.soc[ev, requests[ev]['e_t']] >= requests[ev]['e_soc']
    model.target_con = aml.Constraint(model.EVs, rule=target_soc_rule)

    # Aktuell: Gleichzeitig Laden und Entladen möglich. 
    # Soft-constrained, da in der Regel ineffizient und nicht rentabel.

    # --- Lösung ---
    solver = SolverFactory('gurobi')
    solver.solve(model)
    # for t in timesteps:
    #    print(f"{t:<5} | {aml.value(model.soc[0,t]):>8.1f}% | {aml.value(model.soc[1,t]):>8.1f}% | # {price_forecast[t]:>6}")

    p_ch = [sum([aml.value(model.p_ch[ev, t]) for ev in model.EVs]) * delta_t for t in model.T]
    p_ds = [sum([aml.value(model.p_ds[ev, t]) for ev in model.EVs]) * delta_t for t in model.T]

    p_aggregator = np.array(p_ch) - np.array(p_ds)

    return p_aggregator

In [7]:
from assume.units.storage import Storage
import pandas as pd


class RestrictedStorage(Storage):
    def __init__(self, soc_pivots: pd.DataFrame, **kwargs):
        """ Storage which has to meet given SoC pivots at respective time steps. 
        
        Args:
            soc_pivots (pd.DataFrame): SoC pivots (0.0-1.0) indexed by 
                respective time steps. Storage must be available at these.

        Raises:
            ValidationError: If pivots are not in the range [0.0, 1.0], time
                steps are not valid, or if pivots are not given when storage
                becomes available.
                
        """

        super().__init__(**kwargs)
        
        # ToDo: Add validation.
        
        self.soc_pivots = soc_pivots


In [ ]:
import pyomo.environ as aml
from pyomo.opt import SolverFactory
from assume.strategies.portfolio_strategies import UnitOperatorStrategy


class ArbitrageWithTarget(UnitOperatorStrategy):
    def optimal_schedule(evs, timesteps, requests, price_forecast):
        # --- Konfiguration ---
        delta_t = 1.0 
        eff = 0.93  # 90% Effizienz (gilt für Laden UND Entladen)

        # --- Daten-Setup ---

        model = aml.ConcreteModel()
        model.EVs = aml.Set(initialize=evs.keys())
        model.T = aml.Set(initialize=timesteps)

        # Variablen: Aufgeteilt in Charge und Discharge (beide >= 0)
        model.p_ch = aml.Var(model.EVs, model.T, domain=aml.NonNegativeReals)
        model.p_ds = aml.Var(model.EVs, model.T, domain=aml.NonNegativeReals)
        model.soc = aml.Var(model.EVs, model.T, domain=aml.NonNegativeReals, bounds=(0, 100))

        # --- Zielfunktion ---
        # Kosten = (Gekaufte Energie - Verkaufte Energie) * Preis
        def obj_rule(model):
            return sum((model.p_ch[ev, t] - model.p_ds[ev, t]) * delta_t * price_forecast[t] 
                    for ev in model.EVs for t in model.T)
        model.obj = aml.Objective(rule=obj_rule, sense=aml.minimize)

        # --- Nebenbedingungen ---

        # 1. Leistungslimits (kombiniert für beide Richtungen)
        def max_p_rule(model, ev, t):
            return model.p_ch[ev, t] + model.p_ds[ev, t] <= evs[ev]['p_max']
        model.max_p_con = aml.Constraint(model.EVs, model.T, rule=max_p_rule)

        # 2. Zeitfenster
        def window_rule(model, ev, t):
            if t < requests[ev]['s_t'] or t > requests[ev]['e_t']:
                return (model.p_ch[ev, t] + model.p_ds[ev, t]) == 0
            return aml.Constraint.Skip
        model.window_con = aml.Constraint(model.EVs, model.T, rule=window_rule)

        # 3. SoC Dynamik mit Verlusten
        def soc_dynamic_rule(model, ev, t):
            # Energie-Änderung in %: (Laden * Effizienz - Entladen / Effizienz)
            soc_gain = (model.p_ch[ev, t] * eff - model.p_ds[ev, t] / eff) * delta_t / evs[ev]['cap'] * 100
            
            if t == requests[ev]['s_t']:
                return model.soc[ev, t] == requests[ev]['s_soc'] + soc_gain
            elif t > requests[ev]['s_t'] and t <= requests[ev]['e_t']:
                return model.soc[ev, t] == model.soc[ev, t-1] + soc_gain
            else:
                if t == 0: return model.soc[ev, t] == requests[ev]['s_soc']
                return model.soc[ev, t] == model.soc[ev, t-1]
        model.soc_con = aml.Constraint(model.EVs, model.T, rule=soc_dynamic_rule)

        # 4. Ziel-SoC
        def target_soc_rule(model, ev):
            return model.soc[ev, requests[ev]['e_t']] >= requests[ev]['e_soc']
        model.target_con = aml.Constraint(model.EVs, rule=target_soc_rule)

        # --- Lösung ---
        solver = SolverFactory('gurobi')
        solver.solve(model)
        # for t in timesteps:
        #    print(f"{t:<5} | {aml.value(model.soc[0,t]):>8.1f}% | {aml.value(model.soc[1,t]):>8.1f}% | # {price_forecast[t]:>6}")

        p_ch = [sum([aml.value(model.p_ch[ev, t]) for ev in model.EVs]) * delta_t for t in model.T]
        p_ds = [sum([aml.value(model.p_ds[ev, t]) for ev in model.EVs]) * delta_t for t in model.T]

        p_aggregator = np.array(p_ch) - np.array(p_ds)

        return p_aggregator


    def calculate_bids(self, units_operator, market_config, product_tuples, **kwargs):

        evs = [*units_operator.units.keys()]

        schedule = ...
        max_price, min_price = 3000., 0.
        bids = list()
        for t in range(len(schedule)):
            if schedule[t] > 0:
                bid = (t, float(schedule[t]), max_price)
            elif schedule[t] < 0:
                bid = (t, float(schedule[t]), min_price)
            bids.append(bid)
        
        return bids
    
    


    def optimize(self, market_forecast, grid_fees, capacity_requests, residual_load_hh1, residual_load_hh2):
        # Implement optimization logic here.
        # For demonstration, we will return a dummy schedule.
        return np.zeros((len(capacity_requests), len(market_forecast)))

In [9]:
# Market Clearing
import logging
from datetime import datetime, timedelta

import pandas as pd
from dateutil import rrule as rr

from assume import World
from assume.common.forecaster import DemandForecaster, PowerplantForecaster
from assume.common.market_objects import MarketConfig, MarketProduct


log = logging.getLogger(__name__)

db_uri = "sqlite:///local_db/assume_db.db"

world = World(database_uri=db_uri)

start = datetime(2023, 1, 1)
end = start + timedelta(hours=12)
index = pd.date_range(
    start=start,
    end=end,
    freq="h",
)
simulation_id = "world_script_simulation"

world.setup(
    start=start,
    end=end,
    save_frequency_hours=48,
    simulation_id=simulation_id,
    index=index,
)

marketdesign = [
    MarketConfig(
        market_id="Intraday",
        opening_hours=rr.rrule(rr.HOURLY, interval=12, dtstart=start, until=end),
        opening_duration=timedelta(hours=1),
        market_mechanism="pay_as_clear",
        market_products=[MarketProduct(timedelta(hours=1), 12, timedelta(hours=1))],
    )
]

mo_id = "market_operator"
world.add_market_operator(id=mo_id)

for market_config in marketdesign:
    world.add_market(market_operator_id=mo_id, market_config=market_config)

world.add_unit_operator("demand_operator")

demand_forecast = DemandForecaster(index, demand=schedule)

world.add_unit(
    id="aggregator",
    unit_type="demand",
    unit_operator_id="demand_operator",
    unit_params={
        "min_power": 0,
        "max_power": -1000,
        "bidding_strategies": {"EOM": "demand_energy_naive"},
        "technology": "demand",
    },
    forecaster=demand_forecast,
)

world.add_unit_operator("unit_operator")

nuclear_forecast = PowerplantForecaster(
    index, availability=1, fuel_prices={"uranium": 3, "co2": 0.1}
)

world.add_unit(
    id="nuclear_unit",
    unit_type="power_plant",
    unit_operator_id="unit_operator",
    unit_params={
        "min_power": 200,
        "max_power": 1000,
        "fuel_type": "uranium",
        "bidding_strategies": {"EOM": "powerplant_energy_naive"},
        "technology": "nuclear",
    },
    forecaster=nuclear_forecast,
)

world.run()

INFO:assume.world:Connected to the database


ModuleNotFoundError: No module named 'nest_asyncio2'

In [ ]:

# Calculate KPIs.